# Build `nmdc_metadata.biosample_to_annotation_run`

Precomputes the (biosample, annotation workflow run) mapping with provenance
flags for each MaterialProcessing type in the chain.

See `docs/biosample_to_annotation_run.md` for schema, example queries, and
maintenance instructions — especially what to do when new MaterialProcessing
subclasses are added to the NMDC schema.

**Run this notebook after each NMDC data load.**

### Session setup

In [13]:
import pandas as pd
import time

conn = get_trino_connection()
print("Trino connection ready")

Trino connection ready


### Configuration

**MAINTENANCE NOTE:** `PROCESSING_TYPES` is hardcoded against the MaterialProcessing
subclasses present in BERDL as of 2026-04-30. If a new subclass is added to the
NMDC schema and loaded into BERDL, add it here before rebuilding.

To check for new types: run the preflight cell below and compare its output to
the keys of `PROCESSING_TYPES`.

In [14]:
PROCESSING_TYPES = {
    'nmdc:Extraction':                        'has_extraction',
    'nmdc:LibraryPreparation':                'has_library_prep',
    'nmdc:SubSamplingProcess':                'has_subsampling',
    'nmdc:Pooling':                           'has_pooling',
    'nmdc:ChromatographicSeparationProcess':  'has_chromatographic_separation',
    'nmdc:DissolvingProcess':                 'has_dissolving',
    'nmdc:ChemicalConversionProcess':         'has_chemical_conversion',
    'nmdc:FiltrationProcess':                 'has_filtration',
}

ANNOTATION_WORKFLOW_TYPES = (
    'nmdc:MetagenomeAnnotation',
    'nmdc:MetaproteomicsAnalysis',
    'nmdc:MetabolomicsAnalysis',
    'nmdc:NOMAnalysis',
    'nmdc:MetagenomeAssembly',
    'nmdc:MetatranscriptomeAnnotation',
)

MAX_DEPTH     = 15
TARGET_SCHEMA = 'nmdc_metadata'
TARGET_TABLE  = 'biosample_to_annotation_run'

### Preflight: check for unknown MaterialProcessing types

If any type appears here that is not in `PROCESSING_TYPES`, add it to the
config cell above before proceeding.

In [15]:
cur = conn.cursor()
cur.execute("""
    SELECT type, COUNT(*) AS n
    FROM   nmdc_metadata.material_processing_set
    GROUP  BY type
    ORDER  BY n DESC
""")
observed = pd.DataFrame(cur.fetchall(), columns=["type", "n"])

unknown = set(observed["type"]) - set(PROCESSING_TYPES.keys())
if unknown:
    print(f"WARNING: unknown MaterialProcessing types not in PROCESSING_TYPES:")
    for t in sorted(unknown):
        print(f"  {t}")
    print("Add them to PROCESSING_TYPES before continuing.")
else:
    print("OK: all MaterialProcessing types accounted for")
observed

OK: all MaterialProcessing types accounted for


,type,n
0,nmdc:Extraction,6732
1,nmdc:LibraryPreparation,3123
2,nmdc:SubSamplingProcess,2270
3,nmdc:Pooling,2164
4,nmdc:ChromatographicSeparationProcess,1873
5,nmdc:DissolvingProcess,1718
6,nmdc:ChemicalConversionProcess,1042
7,nmdc:FiltrationProcess,650


### Graph edge table and iterative walk

`nmdc_metadata.graph_edges` is a precomputed 2-column table (src, next_id)
combining the four Silver provenance side tables. It is created once in this
notebook's Step 0 and must be refreshed whenever NMDC data is reloaded.

The walk is iterative — one flat JOIN per hop level — rather than recursive.
`WITH RECURSIVE` in Trino exceeds the 150-stage query planner limit for
datasets of this size regardless of batch configuration.

In [16]:
def _walk_iterative(cur, workflow_ids, max_depth=MAX_DEPTH):
    """Walk from workflow run IDs upstream to biosamples via flat per-hop JOINs.

    Returns (bsm_df, mp_df):
      bsm_df: DataFrame(workflow_run_id, biosample_id, n_hops)
      mp_df:  DataFrame(workflow_run_id, processing_type) — deduplicated
    """
    frontier = pd.DataFrame({'origin': workflow_ids, 'id': workflow_ids})
    bsm_parts = []
    mp_parts  = []

    for depth in range(max_depth):
        if frontier.empty:
            break

        active_ids_sql = ', '.join(f"'{i}'" for i in frontier['id'].unique())

        # One hop: look up all outgoing edges from the current frontier
        cur.execute(f"""
            SELECT src, next_id
            FROM   nmdc_metadata.graph_edges
            WHERE  src IN ({active_ids_sql})
        """)
        edges = pd.DataFrame(cur.fetchall(), columns=['src', 'next_id'])
        if edges.empty:
            break

        next_step = (frontier
                     .merge(edges, left_on='id', right_on='src')
                     [['origin', 'next_id']]
                     .rename(columns={'next_id': 'id'}))

        # Collect biosample endpoints reached this hop
        bsm = next_step[next_step['id'].str.startswith('nmdc:bsm')].copy()
        if not bsm.empty:
            bsm['n_hops'] = depth + 1
            bsm_parts.append(bsm)

        # Continue frontier with non-biosample nodes
        frontier = next_step[~next_step['id'].str.startswith('nmdc:bsm')].drop_duplicates()

        # Resolve MaterialProcessing types for new frontier nodes
        if not frontier.empty:
            new_ids_sql = ', '.join(f"'{i}'" for i in frontier['id'].unique())
            cur.execute(f"""
                SELECT id, type
                FROM   nmdc_metadata.material_processing_set
                WHERE  id IN ({new_ids_sql})
            """)
            mp_rows = cur.fetchall()
            if mp_rows:
                mp_df = pd.DataFrame(mp_rows, columns=['id', 'type'])
                mp_step = (frontier
                           .merge(mp_df, on='id')
                           [['origin', 'type']]
                           .rename(columns={'origin': 'workflow_run_id', 'type': 'processing_type'})
                           .drop_duplicates())
                mp_parts.append(mp_step)

    bsm_out = (
        pd.concat(bsm_parts, ignore_index=True)
          .rename(columns={'origin': 'workflow_run_id', 'id': 'biosample_id'})
          .groupby(['workflow_run_id', 'biosample_id'], as_index=False)['n_hops'].min()
    ) if bsm_parts else pd.DataFrame(columns=['workflow_run_id', 'biosample_id', 'n_hops'])

    mp_out = (
        pd.concat(mp_parts, ignore_index=True).drop_duplicates()
    ) if mp_parts else pd.DataFrame(columns=['workflow_run_id', 'processing_type'])

    return bsm_out, mp_out

### Step 0: build/refresh `nmdc_metadata.graph_edges`

Small (87K rows) precomputed edge table combining the four Silver provenance
side tables. Refresh this whenever NMDC Silver tables are reloaded.

In [17]:
spark = get_spark_session(app_name="build_biosample_to_annotation_run", tenant_name="nmdc")

spark.sql("""
    CREATE OR REPLACE TABLE nmdc_metadata.graph_edges
    USING DELTA AS
    SELECT parent_id AS src, was_informed_by AS next_id, 'was_informed_by' AS slot
    FROM   nmdc_metadata.workflow_execution_set_was_informed_by
    UNION ALL
    SELECT parent_id AS src, has_input AS next_id, 'has_input' AS slot
    FROM   nmdc_metadata.data_generation_set_has_input
    UNION ALL
    SELECT has_output AS src, parent_id AS next_id, 'has_output' AS slot
    FROM   nmdc_metadata.material_processing_set_has_output
    UNION ALL
    SELECT parent_id AS src, has_input AS next_id, 'has_input' AS slot
    FROM   nmdc_metadata.material_processing_set_has_input
""")
n = spark.sql("SELECT COUNT(*) FROM nmdc_metadata.graph_edges").collect()[0][0]
print(f"graph_edges: {n:,} rows")

graph_edges: 87,617 rows


### Step 1: collect annotation workflow runs

In [18]:
t0 = time.monotonic()
cur = conn.cursor()
types_sql = ", ".join(f"'{t}'" for t in ANNOTATION_WORKFLOW_TYPES)
cur.execute(f"""
    SELECT id, type
    FROM   nmdc_metadata.workflow_execution_set
    WHERE  type IN ({types_sql})
""")
wf_df = pd.DataFrame(cur.fetchall(), columns=["workflow_run_id", "workflow_type"])
print(f"{len(wf_df)} annotation workflow runs  ({time.monotonic()-t0:.1f}s)")
wf_df["workflow_type"].value_counts()

12512 annotation workflow runs  (0.2s)


workflow_type
nmdc:MetagenomeAnnotation           4729
nmdc:MetagenomeAssembly             4405
nmdc:MetabolomicsAnalysis           3141
nmdc:MetaproteomicsAnalysis          168
nmdc:MetatranscriptomeAnnotation      69
Name: count, dtype: int64

### Step 2: iterative walk — all workflow runs in one pass

In [19]:
r2b, mp_all = _walk_iterative(cur, wf_df["workflow_run_id"].tolist())
print(f"{len(r2b)} (workflow_run_id, biosample_id) pairs")
print(f"{len(mp_all)} (workflow_run_id, processing_type) hits")
print(f"({time.monotonic()-t0:.1f}s elapsed)")

20132 (workflow_run_id, biosample_id) pairs
26020 (workflow_run_id, processing_type) hits
(5.7s elapsed)


### Step 3: pivot processing types to boolean columns

In [20]:
mp_pivot = (
    mp_all.assign(flag=True)
          .pivot_table(index="workflow_run_id", columns="processing_type",
                       values="flag", aggfunc="any", fill_value=False)
          .reset_index()
)
mp_pivot.columns.name = None
mp_pivot = mp_pivot.rename(columns=PROCESSING_TYPES)
for col in PROCESSING_TYPES.values():
    if col not in mp_pivot.columns:
        mp_pivot[col] = False

result = (
    r2b
    .merge(wf_df, on="workflow_run_id", how="left")
    .merge(mp_pivot[["workflow_run_id"] + list(PROCESSING_TYPES.values())],
           on="workflow_run_id", how="left")
)
for col in PROCESSING_TYPES.values():
    result[col] = result[col].fillna(False)

bool_cols = list(PROCESSING_TYPES.values())
result = result[["biosample_id", "workflow_run_id", "workflow_type", "n_hops"] + bool_cols]

print(f"{len(result)} rows, {len(result.columns)} columns")
result.head()

20132 rows, 12 columns


,workflow_run_id,biosample_id,n_hops,workflow_type,has_extraction,has_library_prep,has_subsampling,has_pooling,has_chromatographic_separation,has_dissolving,has_chemical_conversion,has_filtration
0,nmdc:wfmb-11-00wqq720.1,nmdc:bsm-11-ww5h0770,8,nmdc:MetabolomicsAnalysis,True,False,True,False,False,True,False,False
1,nmdc:wfmb-11-01gz2g47.1,nmdc:bsm-11-1t3pb707,8,nmdc:MetabolomicsAnalysis,True,False,True,False,False,True,False,False
2,nmdc:wfmb-11-01vanz40.1,nmdc:bsm-11-sbxbyj04,10,nmdc:MetabolomicsAnalysis,True,False,True,False,False,False,True,False
3,nmdc:wfmb-11-026xhb37.1,nmdc:bsm-11-pdcjjc38,8,nmdc:MetabolomicsAnalysis,True,False,True,False,False,True,False,False
4,nmdc:wfmb-11-04q4dp82.1,nmdc:bsm-11-6s6nr025,8,nmdc:MetabolomicsAnalysis,True,False,True,False,False,True,False,False


### Step 4: write directly to Silver as a Delta table

In [21]:
(spark.createDataFrame(result)
      .write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable(f"{TARGET_SCHEMA}.{TARGET_TABLE}"))

print(f"Wrote {TARGET_SCHEMA}.{TARGET_TABLE}")
print(f"Total build time: {time.monotonic()-t0:.1f}s")

Wrote nmdc_metadata.biosample_to_gene
Total build time: 8.0s


### Step 5: verify

In [22]:
spark.sql(f"""
    SELECT workflow_type,
           COUNT(DISTINCT biosample_id) AS n_biosamples,
           COUNT(DISTINCT workflow_run_id) AS n_workflow_runs,
           ROUND(AVG(n_hops), 1) AS avg_hops,
           SUM(CAST(has_pooling AS INT)) AS pooled_pairs,
           SUM(CAST(has_extraction AS INT)) AS extracted_pairs
    FROM   {TARGET_SCHEMA}.{TARGET_TABLE}
    GROUP  BY workflow_type
    ORDER  BY n_workflow_runs DESC
""").toPandas()

,workflow_type,n_biosamples,n_workflow_runs,avg_hops,pooled_pairs,extracted_pairs
0,nmdc:MetagenomeAnnotation,7512,4729,6.7,6069,7133
1,nmdc:MetagenomeAssembly,7913,4405,6.6,5501,6432
2,nmdc:MetabolomicsAnalysis,1315,3141,8.6,0,3001
3,nmdc:MetaproteomicsAnalysis,72,168,8.0,0,168
4,nmdc:MetatranscriptomeAnnotation,69,69,2.0,0,0
